In [1]:
import os
import glob
import json
import pandas as pd

def parse_version_name(version_str):
    """从复杂的文件夹名称中解析出具有对比价值的特征维度"""
    # 提取 Base (V8 / V11)
    base = "V11" if "v11" in version_str else "V8" if "v8" in version_str else "Baseline"
    
    # 提取 Scheme (S4-S8)
    scheme = "Unknown"
    for s in ["s4", "s5", "s6", "s7", "s8"]:
        if f"_{s}" in version_str or version_str.endswith(s):
            scheme = s.upper()
            break
            
    # 提取对齐方式 (Lift / Z-Score / Rank / None)
    align = "Raw_Prob" # V8 默认
    if "lift" in version_str: align = "Lift"
    elif "zscore" in version_str: align = "Z-Score"
    elif "rank" in version_str: align = "Rank"
    elif "none" in version_str: align = "None"
    
    # 提取温度系数
    temp = "1.0" # 默认温度
    if "temp" in version_str:
        try:
            temp = version_str.split("temp")[-1]
        except:
            pass
            
    return base, scheme, align, temp

def main():
    # 你的结果落盘根目录
    base_dir = "/NAS/shith/uplift/results/criteo/train_y/TARNET"
    
    # 使用 glob 递归匹配所有跑完的 final_metrics.json
    search_pattern = os.path.join(base_dir, "*/*/final_metrics.json")
    json_files = glob.glob(search_pattern)
    
    if not json_files:
        print(f"⚠️ 在 {base_dir} 下没有找到任何 final_metrics.json 文件，请检查路径或等待实验落盘。")
        return

    records = []
    for filepath in json_files:
        # 从路径中推断出实验版本名，例如 y_v11_s6_lift_temp0.5
        # 路径结构大致为 .../TARNET/<version>/run_<version>/final_metrics.json
        version_dir = os.path.basename(os.path.dirname(os.path.dirname(filepath)))
        
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                metrics = json.load(f)
        except Exception as e:
            print(f"⚠️ 跳过损坏或未完成的文件: {filepath}")
            continue
            
        base, scheme, align, temp = parse_version_name(version_dir)
        
        # 核心指标提取
        records.append({
            "Experiment": version_dir,
            "Base": base,
            "Scheme": scheme,
            "Align": align,
            "Temp": temp,
            "Test_Y_AUUC": metrics.get("Test_Target_Y_AUUC", 0.0),
            "Test_Y_AUQC": metrics.get("Test_Target_Y_AUQC", 0.0),
            "Test_C_AUUC": metrics.get("Test_Target_C_AUUC", 0.0),
            "Test_Y_Lift@10": metrics.get("Test_Target_Y_Lift@10", 0.0)
        })
        
    df = pd.DataFrame(records)
    
    if df.empty:
        print("没有成功解析到任何数据。")
        return

    # 设置 Pandas 显示格式，对齐学术界审美
    pd.set_option('display.max_rows', 100)
    pd.set_option('display.max_columns', 15)
    pd.set_option('display.width', 200)
    pd.set_option('display.float_format', '{:.4f}'.format)

    print("="*100)
    print("🏆 全局 AUUC 终极排行榜 (按 Test_Y_AUUC 倒序)")
    print("="*100)
    df_global = df.sort_values(by="Test_Y_AUUC", ascending=False).reset_index(drop=True)
    # 把最重要的列往前放
    cols = ["Test_Y_AUUC", "Test_C_AUUC", "Test_Y_AUQC", "Base", "Scheme", "Align", "Temp", "Experiment"]
    print(df_global[cols].to_string())
    print("\n\n")

    print("="*100)
    print("📊 分组消融对比 (按 Base -> Scheme -> Test_Y_AUUC 倒序排布)")
    print("="*100)
    # 分组排序能一眼看出同一个架构下，哪种配置最能打
    df_grouped = df.sort_values(by=["Base", "Scheme", "Test_Y_AUUC"], ascending=[True, True, False]).reset_index(drop=True)
    print(df_grouped[cols].to_string())
    
    # 自动落盘成 CSV，方便后续画图或导入 Excel
    output_csv = "ablation_summary.csv"
    df_grouped[cols].to_csv(output_csv, index=False)
    print(f"\n✅ 分析结果已保存至当前目录: {output_csv}")

if __name__ == "__main__":
    main()

🏆 全局 AUUC 终极排行榜 (按 Test_Y_AUUC 倒序)
    Test_Y_AUUC  Test_C_AUUC  Test_Y_AUQC      Base   Scheme     Align   Temp                     Experiment
0        0.9195       0.8264       0.9223        V8       S6  Raw_Prob   10.0               y_v8_s6_temp10.0
1        0.9186       0.8285       0.9213       V11       S6      Rank    5.0          y_v11_s6_rank_temp5.0
2        0.9147       0.8275       0.9180        V8  Unknown  Raw_Prob    1.0                    y_v8_s1_t10
3        0.9133       0.8330       0.9165  Baseline  Unknown  Raw_Prob    1.0                 y_v7_soft_top5
4        0.9123       0.8331       0.9156       V11       S6   Z-Score    5.0        y_v11_s6_zscore_temp5.0
5        0.9119       0.8132       0.9144  Baseline  Unknown  Raw_Prob    1.0             y_v7_conflict_gold
6        0.9113       0.8347       0.9144        V8       S6  Raw_Prob   20.0               y_v8_s6_temp20.0
7        0.9113       0.8096       0.9139        V8       S4  Raw_Prob    1.0                

: 